# Fan-plot NRMSE-cut comparison — all zips at once

**Usage**: edit `NRMSE_MAX` in the parameter cell (optionally `ZIPS` / `N_CURVES`), then rerun it and the cells below.
One figure per zip: left column = no cut (fit_ok only), right column = with `NRMSE <= NRMSE_MAX`, channel by channel.

Reads only the fit checkpoints (~169 MB total); they are loaded once and kept in memory,
so changing the threshold and rerunning takes only tens of seconds.

In [1]:
import os, pickle, sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, '/users/9/li004628/urop/snolab/lp_fit_align/scripts')
from lp_fit_align import ALL_CHANS, CKPT_DIR, RISE_REF_IDX, SAMPLERATE, X_FULL, two_exp_free_pt
print('ready')

ModuleNotFoundError: No module named 'scipy'

In [ ]:
# ---- load fit parameters for all zips (run once, ~10 s) ----
def load_fits(det):
    d = os.path.join(CKPT_DIR, f'zip{det}')
    out = {c: [] for c in ALL_CHANS}
    if not os.path.isdir(d):
        return out
    for f in sorted(os.listdir(d)):
        if f.endswith('_fit.pkl'):
            fits = pickle.load(open(os.path.join(d, f), 'rb'))['fits']
            for c in ALL_CHANS:
                out[c] += [fp for fp in (fits.get(c) or []) if fp is not None and fp['fit_ok']]
    return out

AVAILABLE = sorted(int(x[3:]) for x in os.listdir(CKPT_DIR) if x.startswith('zip') and os.listdir(os.path.join(CKPT_DIR, x)))
FITS = {det: load_fits(det) for det in AVAILABLE}
print('loaded zips:', AVAILABLE)

In [ ]:
# ==== EDIT HERE ====
NRMSE_MAX = 0.4          # threshold to try
ZIPS      = AVAILABLE    # or a subset, e.g. [7, 1, 18]
N_CURVES  = 120          # curves drawn per channel

In [ ]:
# ---- all zips: left = no cut, right = NRMSE <= NRMSE_MAX ----
lo, hi = RISE_REF_IDX - 500, RISE_REF_IDX + 5000
x = X_FULL[lo:hi]; t_ms = x / SAMPLERATE * 1e3

def draw_fan(ax, fps, title):
    rng = np.random.default_rng(0)
    if fps:
        for i in rng.choice(len(fps), min(N_CURVES, len(fps)), replace=False):
            fp = fps[i]
            y = two_exp_free_pt(x, fp['amp'], fp['t_rise'], fp['t_fall'], 0.0, float(RISE_REF_IDX))
            pk = y.max()
            if pk > 0:
                ax.plot(t_ms, y/pk, lw=0.4, alpha=0.25, color='steelblue')
    ax.set_title(f'{title} (n={len(fps)})', fontsize=8)
    ax.grid(alpha=0.2); ax.tick_params(labelsize=7)

for det in ZIPS:
    fits = FITS[det]
    chans = [c for c in ALL_CHANS if fits[c]]
    if not chans:
        print(f'zip{det}: no fits'); continue
    fig, axes = plt.subplots(len(chans), 2, figsize=(13, 2.2*len(chans)), squeeze=False)
    fig.suptitle(f'zip{det} — fitted 2-exp curves, common pretrigger 16050 | '
                 f'left: fit_ok only   right: + NRMSE<={NRMSE_MAX}', fontsize=11)
    for row, c in enumerate(chans):
        draw_fan(axes[row,0], fits[c], f'{c}  no cut')
        draw_fan(axes[row,1], [fp for fp in fits[c] if fp['nrmse'] <= NRMSE_MAX], f'{c}  NRMSE<={NRMSE_MAX}')
    plt.tight_layout(); plt.show(); plt.close(fig)  # free memory before next zip

In [ ]:
# ---- appendix: pass-rate table across zips at this threshold ----
print(f'NRMSE <= {NRMSE_MAX}')
print(f'{"zip":>5} {"fit_ok":>9} {"pass":>9} {"pass%":>7}')
for det in ZIPS:
    n = sum(len(v) for v in FITS[det].values())
    p = sum(sum(1 for fp in v if fp['nrmse'] <= NRMSE_MAX) for v in FITS[det].values())
    if n:
        print(f'{det:>5} {n:>9} {p:>9} {100*p/n:>6.1f}%')